In [10]:
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [3]:
courses_raw

[{'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 48},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 460},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 397},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 153},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 249}]

In [12]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1400

In [8]:
documents[1100]

{'id': '9fd5bae8e6',
 'course': 'stock-markets-analytics-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Does building an accurate ML model actually beat simple technical-indicator rules?',
 'answer': 'The course tackles this directly: it first applies hand-written rules using specific technical indicators, then compares them against ML predictions that use those same indicators as features, so you can see whether the model adds value.'}

In [13]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [14]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
            question, 
            boost_dict={"question": 2.0, "section": 0.5},
            filter_dict={'course': 'llm-zoomcamp'},
            num_results=5
        )
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [6]:
def search(question, course='llm-zoomcamp'):
    boost_dict={"question": 2.0, "section": 0.5}
    filter_dict={'course': course}
    
    search_results = index.search(
            question, 
            boost_dict=boost_dict,
            filter_dict=filter_dict,
            num_results=5
        )
    return search_results

In [15]:
search_results = search(question)

In [20]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [18]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [24]:
USER_PROMPT_TEMPLATE = f'''
Question:
{question}

Context:
{context}
'''

In [22]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [27]:
def build_prompt(question, context):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question, 
        context=context
    )
    return prompt.strip()



In [ ]:
USER_PROMPT_TEMPLATE.format(question=question, context=context)

'\nQuestion:\nI just discovered the course. Can I join now?\n\nContext:\nGeneral Course-Related Questions\nQ: I just discovered the course. Can I still join?\nA: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nGeneral Course-Related Questions\nQ: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nA: You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nGeneral Course-Related Questions\nQ: Does the course certificate show the number of course hours?\nA: No. The certificate does not state a total number of hours.\n\nModule 3: Orchestration\nQ: Why do we need orchestration / Kestra — can\'t I just run the code in a notebook?\nA: Notebooks are great for learning and expe

In [28]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

Module 3: Orchestration
Q: Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A: Notebooks are great for learning and experimenting, but real AI w

In [ ]:
from openai import OpenAI
openai_client = OpenAI()

In [ ]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=prompt
)
response.output_text

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [ ]:
print(response.model_dump_json(indent=2))
response.output[0].content[0].text

AttributeError: 'Response' object has no attribute 'model_dump_json'

In [2]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen3-4B-Instruct-2507"
)

def llm(prompt):
    result = generator(
        prompt,
        max_new_tokens=100,
        num_return_sequences=1
    )

    return result[0]["generated_text"]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [3]:
print(llm("What is RAG?"))

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


What is RAG? How is it different from other methods of retrieval?

RAG stands for **Retrieval-Augmented Generation**. It is a technique used in natural language processing (NLP) and artificial intelligence (AI) to improve the quality of generated text by retrieving relevant information from external knowledge sources before generating a response.

---

### 🔍 What is RAG?

RAG combines two key components:

1. **Retrieval**: A system retrieves relevant documents or facts from a database or knowledge base based on


In [4]:
USER_PROMPT_TEMPLATE = """
    {instructions}

    Question:

    {question}

    Context:

    {context}

    Answer:
"""


In [19]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()


def build_prompt(question, context):
    prompt = USER_PROMPT_TEMPLATE.format(
        instructions=INSTRUCTIONS,
        question=question,
        context=context
    )

    return prompt.strip()

context = build_context(search_results)

prompt = build_prompt(
    question,
    context
)

print(prompt)

Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."


    Question:

    I just discovered the course. Can I join now?

    Context:

    General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the num

In [20]:
result = generator(
    prompt,
    max_new_tokens=200,
    num_return_sequences=1
)

answer = result[0]["generated_text"]

print(answer)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."


    Question:

    I just discovered the course. Can I join now?

    Context:

    General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the num

In [24]:
answer.split("Final Answer:")[-1].strip()

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nYes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [25]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

In [26]:
answer = llm(message_history)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [39]:
answer[2]["content"]

'Yes, you can join now, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [42]:
def ask(instructions, user_prompt, model="Qwen/Qwen3-4B-Instruct-2507"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    answer = llm(message_history)

    return answer[2]["content"]

In [40]:
def rag(query, model="Qwen/Qwen3-4B-Instruct-2507"):
    search_results = search(query)
    context = build_context(search_results)
    prompt = build_prompt(query, context)
    answer = ask(INSTRUCTIONS, prompt, model=model)

    return answer

In [43]:
rag_answer = rag(question)
print(rag_answer)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Yes, you can join now, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.
